# HW04 序列模型、RNN、嵌入向量与注意力机制

姓名：李洁琼
学号：20234080120 

说明：本作业按题目要求简单完成理论题和编程题，编程题均给出测试输出。

## 2 序列模型

### 2.1 理论计算题

字符序列为 `ababc`，可得到转移次数：

- `a -> b` 出现 2 次
- `b -> a` 出现 1 次
- `b -> c` 出现 1 次

词汇表为 `{a, b, c}`，所以词汇表大小为：

\[
V=3
\]

从 `b` 出发的总次数为 2。使用拉普拉斯平滑：

\[
P(x|b)=\frac{count(b\rightarrow x)+1}{count(b\rightarrow *)+V}
\]

因此：

\[
P(a|b)=\frac{1+1}{2+3}=\frac{2}{5}=0.4
\]

\[
P(c|b)=\frac{1+1}{2+3}=\frac{2}{5}=0.4
\]

所以答案为：

\[
P('a'|'b')=0.4,\quad P('c'|'b')=0.4
\]

### 2.2 编程题

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 小写并去除标点，只保留字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # 2. 按空格分词
    words = text.split()

    # 3. 构建词汇表，按词频从高到低排序
    counter = Counter(words)
    vocab_words = sorted(counter.keys(), key=lambda w: (-counter[w], w))
    vocab = {word: i for i, word in enumerate(vocab_words)}

    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(len(words) - n + 1):
        features.append(words[i:i+n])
        if i + n < len(words):
            labels.append(words[i+n])
        else:
            labels.append(None)

    return vocab, (features, labels)


text = "The time machine"
vocab, (features, labels) = preprocess_text(text, 2)

print("词汇表：", vocab)
print("特征：", features)
print("标签：", labels)

词汇表： {'machine': 0, 'the': 1, 'time': 2}
特征： [['the', 'time'], ['time', 'machine']]
标签： ['machine', None]


## 3 循环神经网络

### 3.1 理论计算题

线性 RNN 为：

\[
h_t=W_{hh}h_{t-1}+W_{hx}x_t
\]

\[
o_t=W_{oh}h_t
\]

损失函数为：

\[
L=\frac{1}{2}\sum_{t=1}^{T}(o_t-y_t)^2
\]

令：

\[
\delta_t=\frac{\partial L}{\partial h_t}
\]

通过时间反向传播可得：

\[
\delta_t=W_{oh}^T(o_t-y_t)+W_{hh}^T\delta_{t+1}
\]

因此损失对 \(W_{hh}\) 的梯度为：

\[
\frac{\partial L}{\partial W_{hh}}=\sum_{t=1}^{T}\delta_t h_{t-1}^T
\]

展开来看：

\[
\delta_t=\sum_{k=t}^{T}(W_{hh}^T)^{k-t}W_{oh}^T(o_k-y_k)
\]

当 \(W_{hh}\) 的特征值或范数小于 1 时，多次连乘会使梯度逐渐变小，容易出现梯度消失；当其大于 1 时，梯度会不断放大，容易出现梯度爆炸。

### 3.2 编程题

In [2]:
import numpy as np

def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    a = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a)
    cache = (x_t, h_prev, W_hx, W_hh, h_t)
    return h_t, cache

def rnn_step_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, h_t = cache

    da = dh_next * (1 - h_t ** 2)

    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = np.sum(da, axis=0)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


np.random.seed(0)

batch_size = 2
input_size = 3
hidden_size = 4

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.zeros(hidden_size)

h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
dh_next = np.random.randn(batch_size, hidden_size)

dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print("h_t shape:", h_t.shape)
print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (4,)


## 4 高级循环神经网络

### 4.1 理论计算题

深度双向 RNN 有 \(L\) 层，每层隐藏单元数为 \(H\)，输入维度为 \(D\)，输出维度为 \(O\)。

第一层每个方向的参数量为：

\[
DH+H^2+H
\]

因为是双向，所以第一层参数量为：

\[
2(DH+H^2+H)
\]

从第二层开始，每层输入维度为 \(2H\)，每个方向参数量为：

\[
2H\cdot H+H^2+H=3H^2+H
\]

共有 \(L-1\) 层，所以后续层参数量为：

\[
2(L-1)(3H^2+H)
\]

最后输出层输入维度为 \(2H\)，输出维度为 \(O\)，参数量为：

\[
2HO+O
\]

所以总参数量为：

\[
2(DH+H^2+H)+2(L-1)(3H^2+H)+2HO+O
\]

### 4.2 编程题

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True
        )

    def forward(self, X):
        # outputs: (seq_len, batch, 2 * hidden_dim)
        outputs, h_n = self.rnn(X)

        # h_n: (2, batch, hidden_dim)
        h_forward = h_n[-2]
        h_backward = h_n[-1]

        # 拼接最终前向和后向隐藏状态
        final_hidden = torch.cat([h_forward, h_backward], dim=1)

        return outputs, final_hidden


torch.manual_seed(0)

seq_len = 5
batch = 2
input_dim = 3
hidden_dim = 4

X = torch.randn(seq_len, batch, input_dim)

encoder = BiRNNEncoder(input_dim, hidden_dim)
outputs, final_hidden = encoder(X)

print("outputs shape:", outputs.shape)
print("final_hidden shape:", final_hidden.shape)

outputs shape: torch.Size([5, 2, 8])
final_hidden shape: torch.Size([2, 8])


## 5 嵌入向量

### 5.1 理论计算题

Skip-gram 模型中，中心词为 \(w_c\)，上下文词为 \(w_o\)。中心词向量为 \(v_c\)，上下文词向量为 \(u_o\)。

正样本部分为：

\[
\log \sigma(u_o^T v_c)
\]

如果采样 \(K\) 个负样本，负样本词向量为 \(u_{n_k}\)，则负采样目标函数为：

\[
\log \sigma(u_o^T v_c)+\sum_{k=1}^{K}\log \sigma(-u_{n_k}^T v_c)
\]

损失函数取负号：

\[
L=-\log \sigma(u_o^T v_c)-\sum_{k=1}^{K}\log \sigma(-u_{n_k}^T v_c)
\]

负样本通常从噪声分布中采样，常用方法是按照词频的 \(3/4\) 次方采样：

\[
P_n(w)=\frac{f(w)^{3/4}}{\sum_i f(w_i)^{3/4}}
\]

### 5.2 编程题

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_idxs, target_idxs, W, W_out):
    # context_idxs: (batch, context_size)
    # target_idxs: (batch,)
    # W: (V, d)
    # W_out: (d, V)

    context_embeds = W[context_idxs]           # (batch, context_size, d)
    hidden = context_embeds.mean(dim=1)        # (batch, d)
    logits = hidden @ W_out                    # (batch, V)
    loss = F.cross_entropy(logits, target_idxs)

    return loss


torch.manual_seed(0)

V = 10
d = 5
batch = 3
context_size = 4

context_idxs = torch.tensor([
    [1, 2, 3, 4],
    [2, 3, 4, 5],
    [0, 1, 2, 3]
])

target_idxs = torch.tensor([5, 6, 4])

W = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)

loss = cbow_forward_loss(context_idxs, target_idxs, W, W_out)

print("loss:", loss.item())

loss: 1.8075791597366333


## 6 注意力机制

### 6.1 理论计算题

已知：

\[
Q\in R^{2\times 4},\quad K\in R^{3\times 4},\quad V\in R^{3\times 5}
\]

缩放点积注意力公式为：

\[
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

因为 \(d_k=4\)，所以：

\[
\sqrt{d_k}=2
\]

第一步，计算得分矩阵：

\[
S=\frac{QK^T}{2}
\]

其中 \(Q\) 是 \(2\times 4\)，\(K^T\) 是 \(4\times 3\)，所以：

\[
S\in R^{2\times 3}
\]

第二步，对 \(S\) 的每一行做 softmax：

\[
A=softmax(S)
\]

第三步，加权求和：

\[
O=AV
\]

由于 \(A\in R^{2\times 3}\)，\(V\in R^{3\times 5}\)，所以输出：

\[
O\in R^{2\times 5}
\]

最终表达式为：

\[
O=softmax\left(\frac{QK^T}{2}\right)V
\]

### 6.2 编程题

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, d_model = X.shape

        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        # 调整为 (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 0, 2).reshape(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.permute(1, 0, 2).reshape(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.permute(1, 0, 2).reshape(batch, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)

        # 拼接多头
        context = context.transpose(1, 2).reshape(batch, seq_len, d_model)

        output = self.W_o(context)

        # 变回 (seq_len, batch, d_model)
        output = output.permute(1, 0, 2)

        return output


torch.manual_seed(0)

seq_len = 5
batch = 2
d_model = 4

X = torch.randn(seq_len, batch, d_model)

mha = MultiHeadAttention(d_model=4, num_heads=2)
output = mha(X)

print("output shape:", output.shape)

output shape: torch.Size([5, 2, 4])
